# Classificação Estruturada em Português com Gemini 2.5 Flash

Este notebook está configurado para retornar apenas dados em **Português do Brasil**, utilizando o modelo **Gemini 2.5 Flash** e o SDK `google-genai`.

In [ ]:
import os
import time
import json
import PIL.Image
from google import genai
from pydantic import BaseModel, Field
from typing import List, Optional
from dotenv import load_dotenv
from pathlib import Path

## Definição do Esquema em Português

Utilizamos o `Field` do Pydantic para dar instruções explícitas ao modelo sobre o idioma de cada campo.

In [ ]:
class Deteccao(BaseModel):
    categoria: str = Field(description="Categoria do animal em português (gato, cachorro, pássaro, etc)")
    raca: Optional[str] = Field(description="Raça identificada em português. Se não souber, retorne nulo.")
    descricao: str = Field(description="Descrição detalhada do que o animal está fazendo, exclusivamente em português.")

class AnaliseImagem(BaseModel):
    categoria_principal: str = Field(description="Categoria geral da imagem em português.")
    deteccoes: List[Deteccao] = Field(description="Lista de animais detectados.")
    resumo_informativo: str = Field(description="Resumo executivo da imagem, formatado em português do Brasil.")

In [ ]:
load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
print("Cliente configurado. Modelo: Gemini 2.5 Flash")

## Processando Imagens (Output 100% em Português)

O prompt e o esquema forçam o uso exclusivo da língua portuguesa.

In [ ]:
image_folder = Path("imagens")
image_extensions = [".jpg", ".jpeg", ".png", ".webp"]
files = [f for f in image_folder.iterdir() if f.suffix.lower() in image_extensions]

print(f"Iniciando análise de {len(files)} imagens...")

for i, img_path in enumerate(files):
    print(f"\n[{i+1}/{len(files)}] Processando: {img_path.name}")
    
    try:
        img = PIL.Image.open(img_path)
        
        # Chamada com Gemini 2.5 Flash e esquema em PT-BR
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=["Analise esta imagem detalhadamente. Forneça todos os textos estritamente em Português do Brasil.", img],
            config={
                'response_mime_type': 'application/json',
                'response_schema': AnaliseImagem,
            }
        )
        
        dados = response.parsed
        
        print(f"Principal: {dados.categoria_principal}")
        print(f"Resumo: {dados.resumo_informativo}")
        print("Componentes:")
        for det in dados.deteccoes:
            raca_str = f" ({det.raca})" if det.raca else ""
            print(f" - {det.categoria.upper()}{raca_str}: {det.descricao}")
            
        time.sleep(1)
        
    except Exception as e:
        print(f"Erro ao processar {img_path.name}: {e}")